In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
from google.colab import files
yuklenen = files.upload()

In [ ]:
import pandas as pd

train = pd.read_csv("tr_train.csv", keep_default_na=False)
train = train[train["belirsiz"] != "e"].reset_index(drop=True)

In [ ]:
X = train["mesaj"]
y = (train["ikili"] == "spam").astype(int)

In [ ]:
print(train.shape)
print(y.value_counts())
print(train["fold"].value_counts().sort_index())

In [ ]:
from transformers import AutoTokenizer

MODEL = "dbmdz/bert-base-turkish-cased"
tok = AutoTokenizer.from_pretrained(MODEL)

mesaj = X[0]
print(mesaj)
print(tok.tokenize(mesaj))
print(tok(mesaj)["input_ids"])

In [ ]:
import numpy as np

uzunluklar = [len(tok.tokenize(m)) for m in X]
print("medyan:", np.median(uzunluklar))
print("%90, %95, %99:", np.percentile(uzunluklar, [90, 95, 99]))
print("en uzun:", max(uzunluklar))

In [ ]:
MAX_UZUNLUK = 96

kodlanmis = tok(list(X), truncation=True, padding="max_length",
                max_length=MAX_UZUNLUK, return_tensors="pt")

print(kodlanmis["input_ids"].shape)
print(kodlanmis["input_ids"][0])
print(kodlanmis["attention_mask"][0])

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification

cihaz = "cuda"
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(cihaz)

girdi = {k: v[:4].to(cihaz) for k, v in kodlanmis.items()}
with torch.no_grad():
    cikti = model(**girdi)
print(cikti.logits)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

etiketler = torch.tensor(y.to_numpy())
fold_no = 0
egit = torch.tensor((train["fold"] != fold_no).to_numpy())
dene = torch.tensor((train["fold"] == fold_no).to_numpy())

def veri_seti(maske):
    return TensorDataset(kodlanmis["input_ids"][maske],
                         kodlanmis["attention_mask"][maske],
                         etiketler[maske])

egit_yukleyici = DataLoader(veri_seti(egit), batch_size=16, shuffle=True)
dene_yukleyici = DataLoader(veri_seti(dene), batch_size=32)
print(len(egit_yukleyici), len(dene_yukleyici))

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)
EPOCH = 3

for epoch in range(EPOCH):
    model.train()
    toplam_kayip = 0
    for ids, maske, etiket in egit_yukleyici:
        ids, maske, etiket = ids.to(cihaz), maske.to(cihaz), etiket.to(cihaz)
        cikti = model(input_ids=ids, attention_mask=maske, labels=etiket)
        cikti.loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        toplam_kayip += cikti.loss.item()
    print(f"epoch {epoch+1}: ortalama kayıp = {toplam_kayip / len(egit_yukleyici):.3f}")

In [ ]:
from sklearn.metrics import classification_report

model.eval()
tahminler, gercekler = [], []

with torch.no_grad():
    for ids, maske, etiket in dene_yukleyici:
        cikti = model(input_ids=ids.to(cihaz), attention_mask=maske.to(cihaz))
        tahminler += cikti.logits.argmax(dim=1).cpu().tolist()
        gercekler += etiket.tolist()

print(classification_report(gercekler, tahminler, target_names=["meşru", "spam"], digits=3))

In [ ]:
from sklearn.metrics import f1_score

def fold_calistir(fold_no, epoch=3, lr=2e-5, seed=55):
    torch.manual_seed(seed)
    egit = torch.tensor((train["fold"] != fold_no).to_numpy())
    dene = torch.tensor((train["fold"] == fold_no).to_numpy())
    egit_y = DataLoader(veri_seti(egit), batch_size=16, shuffle=True)
    dene_y = DataLoader(veri_seti(dene), batch_size=32)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(cihaz)
    optimizer = AdamW(model.parameters(), lr=lr)

    for _ in range(epoch):
        model.train()
        for ids, maske, etiket in egit_y:
            cikti = model(input_ids=ids.to(cihaz), attention_mask=maske.to(cihaz),
                          labels=etiket.to(cihaz))
            cikti.loss.backward()
            optimizer.step()
            optimizer.zero_grad()

    model.eval()
    tahminler, gercekler = [], []
    with torch.no_grad():
        for ids, maske, etiket in dene_y:
            cikti = model(input_ids=ids.to(cihaz), attention_mask=maske.to(cihaz))
            tahminler += cikti.logits.argmax(dim=1).cpu().tolist()
            gercekler += etiket.tolist()

    del model, optimizer
    torch.cuda.empty_cache()
    return f1_score(gercekler, tahminler)

berturk_skor = []
for k in range(5):
    s = fold_calistir(k)
    berturk_skor.append(s)
    print(f"fold {k}: spam F1 = {s:.3f}")

print("ortalama:", sum(berturk_skor) / 5)

In [ ]:
print([round(s, 3) for s in berturk_skor])

In [ ]:
MODEL = "xlm-roberta-base"
tok = AutoTokenizer.from_pretrained(MODEL)

uzunluklar = [len(tok.tokenize(m)) for m in X]
print("medyan:", np.median(uzunluklar))
print("%99:", np.percentile(uzunluklar, 99))
print("en uzun:", max(uzunluklar))
print(tok.tokenize(X[0]))

In [ ]:
kodlanmis = tok(list(X), truncation=True, padding="max_length",
                max_length=MAX_UZUNLUK, return_tensors="pt")
print(kodlanmis["input_ids"].shape)

xlmr_skor = []
for k in range(5):
    s = fold_calistir(k)
    xlmr_skor.append(s)
    print(f"fold {k}: spam F1 = {s:.3f}")

print("ortalama:", sum(xlmr_skor) / 5)
print([round(s, 3) for s in xlmr_skor])

In [ ]:
from google.colab import files
yuklenen = files.upload()

In [ ]:
uci = pd.read_csv("uci_sms.csv", keep_default_na=False)
print(uci.shape)
print(uci.columns.tolist())
uci.head()

In [ ]:
print(uci["label"].value_counts())

spam = uci[uci["label"] == 1]
ham = uci[uci["label"] == 0].sample(n=len(spam), random_state=55)
en = pd.concat([spam, ham]).sample(frac=1, random_state=55).reset_index(drop=True)

print(en["label"].value_counts())
print(en.shape)

In [ ]:
en_kod = tok(list(en["sms"]), truncation=True, padding="max_length",
             max_length=MAX_UZUNLUK, return_tensors="pt")
tr_kod = kodlanmis

en_etiket = torch.tensor(en["label"].to_numpy())
tr_etiket = torch.tensor(y.to_numpy())

print(en_kod["input_ids"].shape, tr_kod["input_ids"].shape)
print("96'ya sığmayıp kesilen İngilizce mesaj:", int((en_kod["attention_mask"].sum(dim=1) == MAX_UZUNLUK).sum()))

In [ ]:
torch.manual_seed(55)

en_set = TensorDataset(en_kod["input_ids"], en_kod["attention_mask"], en_etiket)
en_yukleyici = DataLoader(en_set, batch_size=16, shuffle=True)

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(cihaz)
optimizer = AdamW(model.parameters(), lr=2e-5)

for epoch in range(3):
    model.train()
    toplam_kayip = 0
    for ids, maske, etiket in en_yukleyici:
        cikti = model(input_ids=ids.to(cihaz), attention_mask=maske.to(cihaz),
                      labels=etiket.to(cihaz))
        cikti.loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        toplam_kayip += cikti.loss.item()
    print(f"epoch {epoch+1}: ortalama kayıp = {toplam_kayip / len(en_yukleyici):.3f}")

In [ ]:
print(type(model).__name__, tr_kod["input_ids"].shape)

In [ ]:
from sklearn.metrics import confusion_matrix

tr_yukleyici = DataLoader(TensorDataset(tr_kod["input_ids"], tr_kod["attention_mask"], tr_etiket),
                          batch_size=32)

model.eval()
tahminler, gercekler = [], []
with torch.no_grad():
    for ids, maske, etiket in tr_yukleyici:
        cikti = model(input_ids=ids.to(cihaz), attention_mask=maske.to(cihaz))
        tahminler += cikti.logits.argmax(dim=1).cpu().tolist()
        gercekler += etiket.tolist()

print(classification_report(gercekler, tahminler, target_names=["meşru", "spam"], digits=3))
print(confusion_matrix(gercekler, tahminler))

In [ ]:
def transfer_calistir(model_adi, epoch=3, lr=2e-5, seed=55):
    torch.manual_seed(seed)
    tk = AutoTokenizer.from_pretrained(model_adi)
    en_k = tk(list(en["sms"]), truncation=True, padding="max_length",
              max_length=MAX_UZUNLUK, return_tensors="pt")
    tr_k = tk(list(X), truncation=True, padding="max_length",
              max_length=MAX_UZUNLUK, return_tensors="pt")

    en_y = DataLoader(TensorDataset(en_k["input_ids"], en_k["attention_mask"], en_etiket),
                      batch_size=16, shuffle=True)
    tr_y = DataLoader(TensorDataset(tr_k["input_ids"], tr_k["attention_mask"], tr_etiket),
                      batch_size=32)

    m = AutoModelForSequenceClassification.from_pretrained(model_adi, num_labels=2).to(cihaz)
    opt = AdamW(m.parameters(), lr=lr)
    for _ in range(epoch):
        m.train()
        for ids, maske, etiket in en_y:
            c = m(input_ids=ids.to(cihaz), attention_mask=maske.to(cihaz), labels=etiket.to(cihaz))
            c.loss.backward()
            opt.step()
            opt.zero_grad()

    m.eval()
    tahmin, gercek = [], []
    with torch.no_grad():
        for ids, maske, etiket in tr_y:
            c = m(input_ids=ids.to(cihaz), attention_mask=maske.to(cihaz))
            tahmin += c.logits.argmax(dim=1).cpu().tolist()
            gercek += etiket.tolist()
    del m, opt
    torch.cuda.empty_cache()
    return f1_score(gercek, tahmin)

del model
print("BERTurk  (sadece İngilizce -> Türkçe):", round(transfer_calistir("dbmdz/bert-base-turkish-cased"), 3))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

vek = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True,
                      min_df=2, lowercase=False)
A = vek.fit_transform(en["sms"])
B = vek.transform(X)

svm = LinearSVC(class_weight="balanced", random_state=55)
svm.fit(A, en["label"])
print("SVM (sadece İngilizce -> Türkçe):", round(f1_score(y, svm.predict(B)), 3))

In [ ]:
import math
from transformers.utils import logging
logging.set_verbosity_error()          # "LOAD REPORT" uyarılarını sustur

etiketler = torch.tensor(y.to_numpy())

def kodla(model_adi):
    tk = AutoTokenizer.from_pretrained(model_adi)
    return tk(list(X), truncation=True, padding="max_length",
              max_length=MAX_UZUNLUK, return_tensors="pt")

def tahmin_et(m, kod, idx):
    m.eval()
    tahmin = []
    idx = torch.tensor(idx)
    with torch.no_grad():
        for i in range(0, len(idx), 64):
            kisim = idx[i:i + 64]
            c = m(input_ids=kod["input_ids"][kisim].to(cihaz),
                  attention_mask=kod["attention_mask"][kisim].to(cihaz))
            tahmin += c.logits.argmax(dim=1).cpu().tolist()
    return tahmin

def egri_noktasi(model_adi, kod, fold_no, oran, seed=55):
    torch.manual_seed(seed)
    np.random.seed(seed)
    fold = train["fold"].to_numpy()
    egit_idx = np.where(fold != fold_no)[0]
    dene_idx = np.where(fold == fold_no)[0]
    egit_idx = np.random.RandomState(seed).choice(egit_idx, int(len(egit_idx) * oran), replace=False)

    epoch = max(3, math.ceil(60 / math.ceil(len(egit_idx) / 16)))

    m = AutoModelForSequenceClassification.from_pretrained(model_adi, num_labels=2).to(cihaz)
    opt = AdamW(m.parameters(), lr=2e-5)
    for _ in range(epoch):
        m.train()
        karisik = torch.tensor(np.random.permutation(egit_idx))
        for i in range(0, len(karisik), 16):
            b = karisik[i:i + 16]
            c = m(input_ids=kod["input_ids"][b].to(cihaz),
                  attention_mask=kod["attention_mask"][b].to(cihaz),
                  labels=etiketler[b].to(cihaz))
            c.loss.backward()
            opt.step()
            opt.zero_grad()

    egit_f1 = f1_score(y.to_numpy()[egit_idx], tahmin_et(m, kod, egit_idx))
    dene_f1 = f1_score(y.to_numpy()[dene_idx], tahmin_et(m, kod, dene_idx))
    del m, opt
    torch.cuda.empty_cache()
    return egit_f1, dene_f1

In [ ]:
BERTURK = "dbmdz/bert-base-turkish-cased"
ORANLAR = [0.1, 0.25, 0.5, 1.0]
sonuc = {}

kod_b = kodla(BERTURK)
for oran in ORANLAR:
    skorlar = [egri_noktasi(BERTURK, kod_b, k, oran) for k in range(5)]
    sonuc[("BERTurk", oran)] = skorlar
    egit_ort = np.mean([s[0] for s in skorlar])
    dene_ort = np.mean([s[1] for s in skorlar])
    print(f"BERTurk | eğitimde %{int(oran*100):>3} veri | eğitim F1 = {egit_ort:.3f} | sınav F1 = {dene_ort:.3f}")

In [ ]:
XLMR = "xlm-roberta-base"

kod_x = kodla(XLMR)
for oran in ORANLAR:
    skorlar = [egri_noktasi(XLMR, kod_x, k, oran) for k in range(5)]
    sonuc[("XLM-R", oran)] = skorlar
    egit_ort = np.mean([s[0] for s in skorlar])
    dene_ort = np.mean([s[1] for s in skorlar])
    print(f"XLM-R | eğitimde %{int(oran*100):>3} veri | eğitim F1 = {egit_ort:.3f} | sınav F1 = {dene_ort:.3f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4.5))
renkler = {"BERTurk": "tab:blue", "XLM-R": "tab:orange"}

for model in ["BERTurk", "XLM-R"]:
    if (model, ORANLAR[0]) not in sonuc:
        continue
    mesaj_sayisi = [int(517 * o) for o in ORANLAR]
    dene = [np.mean([s[1] for s in sonuc[(model, o)]]) for o in ORANLAR]
    dene_std = [np.std([s[1] for s in sonuc[(model, o)]]) for o in ORANLAR]
    egit = [np.mean([s[0] for s in sonuc[(model, o)]]) for o in ORANLAR]
    plt.errorbar(mesaj_sayisi, dene, yerr=dene_std, fmt="o-", color=renkler[model],
                 capsize=3, label=f"{model} sınav (CV)")
    plt.plot(mesaj_sayisi, egit, "o--", color=renkler[model], alpha=0.5,
             label=f"{model} eğitim")

plt.xlabel("eğitimde kullanılan mesaj sayısı")
plt.ylabel("spam F1")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("ogrenme_egrisi_bert.png", dpi=150, bbox_inches="tight")
plt.show()